# NemoClaw TTS Server (Chatterbox on GPU)

This notebook runs the Chatterbox TTS service on a free Colab T4 GPU
and exposes it publicly via ngrok so your local NemoClaw can call it.

**Steps:**
1. Runtime → Change runtime type → T4 GPU
2. Run all cells top to bottom
3. Copy the `export` lines printed at the end into your Mac terminal
4. Restart the local voice server

> **Note:** Free Colab sessions last ~12 hours. Voice enrollments are stored
> in `/tmp` and will be lost when the session ends — users will need to
> re-enroll. Upgrade to Colab Pro for longer sessions.

In [ ]:
# ── Cell 1: Verify GPU ──────────────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    raise RuntimeError('No GPU found. Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── Cell 2: Install dependencies ────────────────────────────────────────────
# Install transformers==5.2.0 first (required by chatterbox-tts)
!pip install -q "transformers==5.2.0"
!pip install -q chatterbox-tts setuptools pyngrok \
    fastapi uvicorn[standard] python-multipart \
    librosa soundfile torchaudio
print('Done.')

In [ ]:
# ── Cell 3: Write the TTS server ────────────────────────────────────────────
server_code = '''
import io, logging, os, shutil, tempfile, uuid
from pathlib import Path
from typing import Optional

import torch, torchaudio
from fastapi import FastAPI, File, Form, Header, HTTPException, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import Response

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

API_KEY   = os.environ.get("TTS_API_KEY", "")
DEVICE    = os.environ.get("TTS_DEVICE", "cuda")
VOICES_DIR = Path("/tmp/tts-voices")
VOICES_DIR.mkdir(parents=True, exist_ok=True)

app = FastAPI(title="NemoClaw TTS")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

model = None
voice_registry: dict[str, Path] = {}

def _check_key(k):
    if API_KEY and k != API_KEY:
        raise HTTPException(401, "Invalid API key")

@app.on_event("startup")
async def startup():
    global model
    from chatterbox.tts import ChatterboxTTS
    logger.info("Loading Chatterbox on %s", DEVICE)
    model = ChatterboxTTS.from_pretrained(device=DEVICE)
    logger.info("Warming up...")
    model.generate("Hello.")
    logger.info("Ready.")

@app.get("/health")
def health():
    return {"status": "ok", "device": DEVICE, "voices": len(voice_registry)}

@app.post("/enroll")
async def enroll(audio: UploadFile = File(...), x_api_key: Optional[str] = Header(None)):
    _check_key(x_api_key)
    raw = await audio.read()
    suffix = Path(audio.filename or "audio.webm").suffix or ".webm"
    with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as f:
        f.write(raw); upload_path = f.name
    voice_id = str(uuid.uuid4())
    ref_path = VOICES_DIR / f"{voice_id}.wav"
    try:
        import librosa, soundfile as sf
        arr, _ = librosa.load(upload_path, sr=24000, mono=True)
        sf.write(str(ref_path), arr, 24000)
        model.prepare_conditionals(str(ref_path))
    finally:
        Path(upload_path).unlink(missing_ok=True)
    voice_registry[voice_id] = ref_path
    logger.info("Enrolled %s", voice_id)
    return {"voice_id": voice_id}

@app.post("/synthesize")
async def synthesize(
    text: str = Form(...),
    voice_id: Optional[str] = Form(None),
    x_api_key: Optional[str] = Header(None),
):
    _check_key(x_api_key)
    if voice_id and voice_id in voice_registry:
        wav = model.generate(text, audio_prompt_path=str(voice_registry[voice_id]))
    else:
        wav = model.generate(text)
    buf = io.BytesIO()
    torchaudio.save(buf, wav, model.sr, format="wav")
    return Response(content=buf.getvalue(), media_type="audio/wav")
'''

with open('/content/server.py', 'w') as f:
    f.write(server_code)
print('server.py written.')

In [ ]:
# ── Cell 4: Set your API key ────────────────────────────────────────────────
import os, secrets

# Change this to any secret string you want, or leave it to auto-generate
MY_API_KEY = "nemoclaw-tts-secret"   # ← change me

os.environ['TTS_API_KEY'] = MY_API_KEY
os.environ['TTS_DEVICE']  = 'cuda'
print(f'API key set: {MY_API_KEY}')

In [ ]:
# ── Cell 5: ngrok auth token (required for stable URLs) ─────────────────────
# Get your free token at https://dashboard.ngrok.com/get-started/your-authtoken
# Paste it below:
NGROK_TOKEN = ""   # ← paste your ngrok token here

from pyngrok import ngrok, conf
if NGROK_TOKEN:
    conf.get_default().auth_token = NGROK_TOKEN
    print('ngrok auth token set.')
else:
    print('⚠️  No ngrok token — URL will work but expires in 2 hours.')
    print('   Get a free token at https://dashboard.ngrok.com/get-started/your-authtoken')

In [ ]:
# ── Cell 6: Start server + tunnel ───────────────────────────────────────────
import subprocess, time, requests
from pyngrok import ngrok

LOG_FILE = '/content/uvicorn.log'

# Start uvicorn, capturing output to a log file
with open(LOG_FILE, 'w') as log:
    proc = subprocess.Popen(
        ['uvicorn', 'server:app', '--host', '0.0.0.0', '--port', '8000'],
        cwd='/content',
        env={**os.environ},
        stdout=log,
        stderr=log,
    )

# Open ngrok tunnel immediately (requests will queue until server is ready)
tunnel = ngrok.connect(8000, 'http')
public_url = tunnel.public_url.replace('http://', 'https://')

# Wait for Chatterbox to finish loading (up to 3 minutes)
print('Waiting for Chatterbox to load (this takes ~60-90 seconds)...')
ready = False
for i in range(36):  # 36 × 5s = 3 minutes
    time.sleep(5)
    # Check if uvicorn crashed
    if proc.poll() is not None:
        print('\n❌ uvicorn crashed! Last log output:')
        with open(LOG_FILE) as f:
            print(f.read()[-3000:])
        raise RuntimeError('uvicorn exited unexpectedly — see log above')
    # Check if server is responding
    try:
        r = requests.get('http://localhost:8000/health', timeout=3)
        if r.status_code == 200:
            ready = True
            break
    except Exception:
        pass
    elapsed = (i + 1) * 5
    print(f'  [{elapsed}s] Still loading...', end='\r')

if not ready:
    print('\n❌ Server did not become ready in 3 minutes. Log output:')
    with open(LOG_FILE) as f:
        print(f.read()[-3000:])
    raise RuntimeError('Timeout waiting for server')

print(f'\n✅ Chatterbox loaded and ready!')
print('\n' + '='*60)
print(f'  TTS Server is LIVE at:\n')
print(f'  {public_url}')
print('='*60)
print('\nRun these in your Mac terminal to use Railway TTS:')
print(f'\nexport RAILWAY_TTS_URL={public_url}')
print(f'export RAILWAY_TTS_API_KEY={MY_API_KEY}')
print('\nThen restart the voice server:')
print('SANDBOX_NAME=maloclaw-assistant bash scripts/start-services.sh')
print('\n(Keep this Colab tab open — closing it stops the server)')

In [ ]:
# ── Cell 7: Keep-alive (run this to prevent Colab idle timeout) ─────────────
# Colab disconnects after ~90 min of inactivity.
# This cell runs forever and prints a heartbeat every 10 minutes.
import time
print('Keep-alive running. Interrupt kernel to stop.')
i = 0
while True:
    time.sleep(600)  # 10 minutes
    i += 1
    # Also check server health
    try:
        r = requests.get('http://localhost:8000/health', timeout=5)
        status = r.json().get('status', '?')
    except Exception as e:
        status = f'ERROR: {e}'
    print(f'[{i*10} min] Server status: {status} | URL: {public_url}')